# Bases de données — Exercice 8 : les films d'IMDb (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue : la requête SQL à taper dans la **console H2**, et le résultat obtenu |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** `RUNSCRIPT` et ordre de chargement imposé par les clés étrangères, `SELECT ... WHERE`, `LIKE`, `COUNT`, `ORDER BY`, `UPDATE`, `INSERT INTO`, `DELETE` et contrainte de référence, jointures sur deux et trois relations. Partie D : fonctions `CAST` et `LENGTH`, sous-requête avec `IN`.

> Le site imdb.com met à disposition des jeux de données rassemblant des informations sur les œuvres audiovisuelles diffusées dans le monde. On a extrait de ces jeux de données un sous-ensemble concernant les films diffusés depuis 2010 (fichier : imdb.zip), structuré selon le schéma relationnel ci-dessous.
>
> - **Movies** (films) : `tconst` (identifiant unique du film), `primaryTitle` (titre du film le plus répandu), `releaseYear` (année de sortie), `runtime` (durée en minutes), `genres` (genre(s) du film, par exemple `'Documentary'`, `'Comedy,Drama,Fantasy'`).
> - **Titles** (traductions des titres) : `tconst` (identifiant du film), `ordering` (clé d'enregistrement), `title` (traduction du titre), `language` (langue du titre), `isOriginal` (`'1'` s'il s'agit du titre original, `'0'` sinon).
> - **Names** (acteurs, réalisateurs, etc.) : `nconst` (identifiant de la personne), `name` (prénom et nom), `birthYear` (année de naissance), `deathYear` (année de décès).
> - **Crews** (créateurs du film) : `tconst` (identifiant du film), `director` (identifiant du réalisateur), `writer` (identifiant de l'auteur).
> - **Principals** (acteurs) : `tconst` (identifiant du film), `ordering` (clé d'enregistrement), `nconst` (identifiant de l'acteur), `character` (nom du personnage joué).
>
> Tous les attributs sont de type CHAR. Dans un enregistrement, un attribut non renseigné a pour valeur `'\N'`.

---
# MISE EN PLACE

En classe, les requêtes SQL sont tapées dans la **console H2** (le SGBD utilisé dans le cours). C'est aussi le cas ici : **chaque requête de ce notebook a été exécutée dans H2**, et le résultat affiché sous la requête est celui qu'a donné H2. Il n'y a **aucun code à exécuter** dans ce notebook : il suffit de recopier les requêtes dans la console H2.

**Pour ouvrir la console H2 :** lancer « H2 Console », se connecter avec la JDBC URL `jdbc:h2:~/nsi`, l'utilisateur `sa` et un mot de passe vide.

> ⚠️ **Liberté prise :** les résultats ont été obtenus avec H2 version 2.5.252 et recopiés tels quels, à trois détails près : la durée d'exécution affichée par H2 (« 12 ms ») a été supprimée, car elle change à chaque fois ; pour les résultats longs, seules les 12 premières lignes sont recopiées, suivies de « ... » et du nombre total de lignes ; dans les messages d'erreur, le nom technique de l'erreur Java (`org.h2.jdbc....Exception`) a été remplacé par « Erreur : ». Dans la console du navigateur, le résultat apparaît sous forme de tableau ; le contenu est le même.

> ⚠️ **À savoir avant de commencer :** la base est **grosse** (près de 200 000 films et 800 000 titres). Une requête peut mettre quelques secondes avant de répondre : il faut attendre que la console rende la main.

> ⚠️ **Attention à l'ordre des questions :** les questions 3 à 6 (partie B) **modifient** la base. Les résultats de la partie C ont été obtenus **après** ces modifications, comme dans l'ordre du sujet.

---
# PARTIE A : création des tables et prise en main du jeu de données

---
# QUESTION 1

#### 📝 ÉNONCÉ


> 1. (a) Télécharger et décompresser l'archive imdb.zip.
>    (b) Exécuter les instructions SQL contenues dans le fichier imdb_tables.sql pour créer les relations.
>    (c) Les autres fichiers contiennent les instructions SQL qui permettent de faire les enregistrements dans chacune des relations : exécuter ces fichiers (dans un ordre compatible avec le schéma relationnel).

#### ✅ RÉSOLUTION


**(a)** L'archive contient six fichiers : `imdb_tables.sql`, `movies.sql`, `names.sql`, `titles.sql`, `principals.sql` et `crews.sql`.

**(b) et (c)** Dans la console H2 (en remplaçant `/chemin/vers/` par le dossier où l'archive a été décompressée), on exécute les fichiers **dans cet ordre** :

```sql
RUNSCRIPT FROM '/chemin/vers/imdb_tables.sql';
RUNSCRIPT FROM '/chemin/vers/movies.sql';
RUNSCRIPT FROM '/chemin/vers/names.sql';
RUNSCRIPT FROM '/chemin/vers/titles.sql';
RUNSCRIPT FROM '/chemin/vers/principals.sql';
RUNSCRIPT FROM '/chemin/vers/crews.sql';
```

*Résultat dans la console H2 (une ligne par fichier) :*

```text
(Update count: 11)
(Update count: 194208)
(Update count: 467748)
(Update count: 796487)
(Update count: 533718)
(Update count: 186359)
```

**L'ordre :** d'abord `Movies` et `Names`, qui ne font référence à aucune autre relation ; ensuite `Titles`, `Principals` et `Crews`, qui font référence à `Movies` (et `Crews` aussi à `Names`).

*Vérification : le nombre de films et de personnes enregistrés.*

```sql
SELECT COUNT(*) FROM Movies; SELECT COUNT(*) FROM Names;
```

*Résultat dans la console H2 :*

```text
COUNT(*)
194207
(1 row)
COUNT(*)
467747
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Pourquoi l'ordre compte :** le fichier `imdb_tables.sql` déclare des **clés étrangères** (partie 10.1 du cours). Par exemple, dans `Crews`, `FOREIGN KEY (tconst) REFERENCES Movies(tconst)` signifie qu'on ne peut pas enregistrer l'équipe d'un film qui n'est **pas encore** dans `Movies` : le SGBD refuse la ligne (partie 10.3, contrainte de référence).

| Relation | Fait référence à | Doit donc être remplie après |
|:---|:---|:---|
| Movies | rien | — |
| Names | rien | — |
| Titles | Movies | Movies |
| Principals | Movies | Movies |
| Crews | Movies et Names | Movies et Names |

- Entre `Movies` et `Names`, l'ordre est libre ; entre `Titles`, `Principals` et `Crews` aussi.
- `imdb_tables.sql` commence par des `DROP TABLE IF EXISTS` : relancer le fichier **efface** les relations et leur contenu. C'est pratique pour repartir de zéro après la partie B.
- Le nombre affiché par `RUNSCRIPT` dépasse d'une unité le nombre de lignes enregistrées (194 208 pour 194 207 films) : c'est une particularité de l'affichage de H2. Pour connaître le contenu réel d'une relation, on utilise `COUNT(*)`.
- Les fichiers sont gros (jusqu'à 60 Mo) : chaque `RUNSCRIPT` peut prendre de quelques secondes à quelques minutes.

---
# QUESTION 2

> 2. Écrire les requêtes SQL permettant d'afficher les réponses aux questions suivantes.

---
# QUESTION 2 (a)

#### 📝 ÉNONCÉ


> (a) Quel est le nombre de films sortis en 2024 ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Movies WHERE releaseYear = '2024';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
88
(1 row)
```

**Réponse : 88 films.**

#### 💡 EXPLICATION PÉDAGOGIQUE


- On **compte** des lignes de `Movies` : `COUNT(*)` (partie 8 du cours), avec un `WHERE` sur l'année.
- **Piège :** tous les attributs sont de type **CHAR** (texte). L'année s'écrit donc **entre guillemets** : `releaseYear = '2024'`. Sans guillemets, H2 doit convertir chaque valeur en nombre. Ici, ça marche par chance, parce que toutes les années sont renseignées. Mais la même écriture échoue sur un attribut qui contient des `'\N'`, comme la durée :

```sql
SELECT COUNT(*) FROM Movies WHERE runtime = 90;
```

*Résultat dans la console H2 :*

```text
Erreur : Erreur lors de la conversion de données "CHARACTER VARYING to DECFLOAT"
Data conversion error converting "CHARACTER VARYING to DECFLOAT"; SQL statement:
SELECT COUNT(*) FROM Movies WHERE runtime = 90 [22018-252]
```

- Le nombre paraît petit pour une année entière : les données ont été extraites d'IMDb **avant** 2024. Les films de 2024 présents dans la base sont ceux qui étaient déjà **annoncés** à ce moment-là.

---
# QUESTION 2 (b)

#### 📝 ÉNONCÉ


> (b) Quel est le nombre de films sortis depuis le début de l'année ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Movies WHERE releaseYear = '2026';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
5
(1 row)
```

**Réponse : 5 films** (pour l'année 2026, celle de l'écriture de cette correction).

#### 💡 EXPLICATION PÉDAGOGIQUE


- La base ne contient que l'**année** de sortie, pas la date exacte. « Depuis le début de l'année » revient donc à compter les films dont l'année de sortie est l'**année en cours**.
- ⚠️ **Liberté prise :** le sujet ne précise pas l'année ; on a pris 2026. Une autre année se traite en changeant seulement la valeur dans le `WHERE`.
- Comme pour la question (a), les films de 2026 présents dans la base sont des films **annoncés** au moment de l'extraction. La base va même jusqu'en 2028 :

```sql
SELECT MAX(releaseYear) FROM Movies;
```

*Résultat dans la console H2 :*

```text
MAX(RELEASEYEAR)
2028
(1 row)
```

---
# QUESTION 2 (c)

#### 📝 ÉNONCÉ


> (c) Quel est le nombre de films sortis en 2025 dont le genre contient le mot-clé 'Comedy' ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Movies WHERE releaseYear = '2025' AND genres LIKE '%Comedy%';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
2
(1 row)
```

**Réponse : 2 films.**

#### 💡 EXPLICATION PÉDAGOGIQUE


- Deux conditions à la fois : l'année **et** le genre, reliées par `AND` (partie 7.2).
- « Dont le genre **contient** » : l'attribut `genres` peut contenir plusieurs genres séparés par des virgules (`'Comedy,Drama'`). On ne peut donc pas écrire `genres = 'Comedy'`, qui ne garderait que les films dont le **seul** genre est la comédie. On cherche le mot **à l'intérieur** du texte avec `LIKE '%Comedy%'` (partie 7.3) : `%` remplace n'importe quelle suite de caractères, avant et après.
- Pour vérifier, on peut afficher les films au lieu de les compter :

```sql
SELECT primaryTitle, genres FROM Movies WHERE releaseYear = '2025' AND genres LIKE '%Comedy%';
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE          | GENRES
Honor Yourself        | Comedy,Drama
Once the Music Played | Comedy,Drama,Musical
(2 rows)
```

---
# QUESTION 2 (d)

#### 📝 ÉNONCÉ


> (d) Quelle est l'année de naissance de Juliette Binoche ?

#### ✅ RÉSOLUTION

```sql
SELECT birthYear FROM Names WHERE name = 'Juliette Binoche';
```

*Résultat dans la console H2 :*

```text
BIRTHYEAR
1964
(1 row)
```

**Réponse : 1964.**

#### 💡 EXPLICATION PÉDAGOGIQUE


- Une seule relation suffit : `Names`, qui contient le nom et l'année de naissance.
- Le nom complet est stocké dans **un seul** attribut, `name` (`'Juliette Binoche'`), et non dans deux attributs « prénom » et « nom ».

---
# QUESTION 2 (e)

#### 📝 ÉNONCÉ


> (e) Quel est le titre original du film intitulé 'Parasite' ?

#### ✅ RÉSOLUTION

```sql
SELECT Titles.title
FROM Movies
JOIN Titles ON Movies.tconst = Titles.tconst
WHERE Movies.primaryTitle = 'Parasite' AND Titles.isOriginal = '1';
```

*Résultat dans la console H2 :*

```text
TITLE
Gisaengchung
(1 row)
```

**Réponse : Gisaengchung** (le titre coréen du film de Bong Joon Ho).

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Le besoin :** le titre « Parasite » est dans `Movies` (attribut `primaryTitle`), mais l'information « titre original » est dans `Titles` (attribut `isOriginal`). Il faut donc **recoller** les deux relations : c'est une **jointure** (partie 10.2).
- **La règle de jointure :** `Titles.tconst` fait référence à `Movies.tconst`, d'où `ON Movies.tconst = Titles.tconst`.
- Après la jointure, on garde la ligne du film « Parasite » **et** celle qui est le titre original : `isOriginal = '1'`, entre guillemets, car c'est du texte.
- Le film lui-même, pour voir son identifiant :

```sql
SELECT * FROM Movies WHERE primaryTitle = 'Parasite';
```

*Résultat dans la console H2 :*

```text
TCONST    | PRIMARYTITLE | RELEASEYEAR | RUNTIME | GENRES
tt6751668 | Parasite     | 2019        | 132     | Comedy,Drama,Thriller
(1 row)
```

- **Pourquoi passer par `Movies` et pas par `Titles` :** dans `Titles`, le titre « Parasite » apparaît pour **deux** films différents. L'un d'eux est un autre film (« Huba », 2014), traduit « Parasite » en Espagne. En partant de `Movies.primaryTitle`, on tombe sur le film le plus connu sous ce titre.

```sql
SELECT DISTINCT tconst FROM Titles WHERE title = 'Parasite';
```

*Résultat dans la console H2 :*

```text
TCONST
tt3522760
tt6751668
(2 rows)
```

---
# PARTIE B : requêtes d'insertion, de mise à jour et de suppression

> Écrire les requêtes permettant de mettre à jour la base de données en tenant compte des informations ci-dessous.

---
# QUESTION 3

#### 📝 ÉNONCÉ


> 3. Le code ISO 'DDDE' (ancienne RDA) a été remplacé par le code 'DE' (Allemagne).

#### ✅ RÉSOLUTION

```sql
UPDATE Titles SET language = 'DE' WHERE language = 'DDDE';
```

*Résultat dans la console H2 :*

```text
(Update count: 1)
```

*Vérifications, avant et après :*

```sql
SELECT * FROM Titles WHERE language = 'DDDE';
```

*Résultat dans la console H2 :*

```text
TCONST     | ORDERING | TITLE         | LANGUAGE | ISORIGINAL
tt12926126 | 3        | After the End | DDDE     | 0
(1 row)
```

```sql
SELECT COUNT(*) FROM Titles WHERE language = 'DDDE';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
0
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- On **modifie** des valeurs existantes : c'est un `UPDATE` (partie 12 du cours). `SET` donne la nouvelle valeur, `WHERE` dit quelles lignes modifier.
- **Où se trouve le code 'DDDE' ?** Dans `Titles`, attribut `language`. Malgré son nom, cet attribut contient le plus souvent un **code de pays** (`'FR'`, `'DE'`, `'PL'`…), qui indique où le titre est utilisé, et non une langue. C'est donc là qu'apparaît le code de l'ancienne RDA.
- **Erreur grave à éviter :** oublier le `WHERE`. `UPDATE Titles SET language = 'DE';` mettrait `'DE'` sur **les 796 486 titres**.
- Bonne habitude : faire un `SELECT` avec le même `WHERE` **avant** l'`UPDATE`, pour voir les lignes qui vont être modifiées.

---
# QUESTION 4

#### 📝 ÉNONCÉ


> 4. L'acteur Udo Kier, né en 944, est mort le 24 novembre 2025 à l'âge de 81 ans.

#### ✅ RÉSOLUTION

```sql
UPDATE Names SET deathYear = '2025' WHERE name = 'Udo Kier';
```

*Résultat dans la console H2 :*

```text
(Update count: 1)
```

*Vérifications, avant et après :*

```sql
SELECT * FROM Names WHERE name = 'Udo Kier';
```

*Résultat dans la console H2 :*

```text
NCONST    | NAME     | BIRTHYEAR | DEATHYEAR
nm0001424 | Udo Kier | 1944      | \N
(1 row)
```

```sql
SELECT * FROM Names WHERE name = 'Udo Kier';
```

*Résultat dans la console H2 :*

```text
NCONST    | NAME     | BIRTHYEAR | DEATHYEAR
nm0001424 | Udo Kier | 1944      | 2025
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Ce qui change dans la base :** seulement l'année de décès. La base ne stocke que des **années** : le jour et le mois (24 novembre) ne peuvent pas être enregistrés.
- L'année de naissance est **déjà** dans la base, et elle vaut 1944. « Né en 944 » est une **coquille** du sujet : avec un décès en 2025 à 81 ans, la naissance est bien en 1944. Il n'y a rien à modifier de ce côté.
- On remplace `'\N'` (« non renseigné ») par `'2025'`, entre guillemets puisque tous les attributs sont du texte.

---
# QUESTION 5

#### 📝 ÉNONCÉ


> 5. Steven Spielberg serait en train de réaliser un film intitulé 'Eternal Life' qui devrait sortir en 2080.

#### ✅ RÉSOLUTION


Il faut **deux** enregistrements : le film dans `Movies`, puis son réalisateur dans `Crews`.

**Étape 1 : trouver l'identifiant de Steven Spielberg.**

```sql
SELECT nconst FROM Names WHERE name = 'Steven Spielberg';
```

*Résultat dans la console H2 :*

```text
NCONST
nm0000229
(1 row)
```

**Étape 2 : choisir un identifiant pour le nouveau film**, et vérifier qu'il n'est pas déjà pris.

```sql
SELECT * FROM Movies WHERE tconst = 'tt9999999';
```

*Résultat dans la console H2 :*

```text
TCONST | PRIMARYTITLE | RELEASEYEAR | RUNTIME | GENRES
(0 rows)
```

**Étape 3 : enregistrer le film, puis son réalisateur.**

```sql
INSERT INTO Movies VALUES ('tt9999999', 'Eternal Life', '2080', '\N', '\N');
```

*Résultat dans la console H2 :*

```text
(Update count: 1)
```

```sql
INSERT INTO Crews VALUES ('tt9999999', 'nm0000229', '\N');
```

*Résultat dans la console H2 :*

```text
(Update count: 1)
```

*Vérification, avec une jointure :*

```sql
SELECT Movies.primaryTitle, Movies.releaseYear, Names.name
FROM Movies
JOIN Crews ON Movies.tconst = Crews.tconst
JOIN Names ON Crews.director = Names.nconst
WHERE Movies.primaryTitle = 'Eternal Life';
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE | RELEASEYEAR | NAME
Eternal Life | 2080        | Steven Spielberg
(1 row)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Le besoin :** l'information « Spielberg réalise ce film » ne se trouve pas dans `Movies`, mais dans `Crews`, qui relie un film (`tconst`) à son réalisateur (`director`). Il faut donc ajouter **une ligne dans chaque** relation.
- ⚠️ **Liberté prise :** le sujet ne donne pas d'identifiant pour le nouveau film. On a choisi `'tt9999999'`, en vérifiant d'abord qu'aucun film ne l'utilise (c'est une **clé primaire**, partie 9 : deux films ne peuvent pas avoir le même identifiant).
- **Valeurs inconnues :** la durée, le genre et l'auteur ne sont pas connus. Le sujet indique qu'un attribut non renseigné vaut `'\N'`.
- **L'ordre compte :** `Crews.tconst` fait référence à `Movies.tconst`. Si on enregistre l'équipe **avant** le film, H2 refuse la ligne :

```sql
INSERT INTO Crews VALUES ('tt9999999', 'nm0000229', '\N');
```

*Résultat dans la console H2 :*

```text
Erreur : Intégrité référentielle violation de contrainte: "CONSTRAINT_3D: PUBLIC.CREWS FOREIGN KEY(TCONST) REFERENCES PUBLIC.MOVIES(TCONST) ('tt9999999')"
Referential integrity constraint violation: "CONSTRAINT_3D: PUBLIC.CREWS FOREIGN KEY(TCONST) REFERENCES PUBLIC.MOVIES(TCONST) ('tt9999999')"; SQL statement:
INSERT INTO Crews VALUES ('tt9999999', 'nm0000229', '\N') [23506-252]
```

- **Pourquoi `'\N'` est accepté pour l'auteur :** `Crews.writer` est aussi une clé étrangère vers `Names`. La valeur `'\N'` devrait donc être refusée… sauf que la base contient une « personne » dont l'identifiant est `'\N'`, justement prévue pour cela :

```sql
SELECT * FROM Names WHERE nconst = '\N';
```

*Résultat dans la console H2 :*

```text
NCONST | NAME | BIRTHYEAR | DEATHYEAR
\N     | \N   | \N        | \N
(1 row)
```

- **Variante en une requête :** au lieu de recopier `'nm0000229'` à la main, on peut écrire la recherche à sa place, entre parenthèses (sous-requête, vue dans la partie 8 du cours) : `INSERT INTO Crews VALUES ('tt9999999', (SELECT nconst FROM Names WHERE name = 'Steven Spielberg'), '\N');`.

---
# QUESTION 6

#### 📝 ÉNONCÉ


> 6. Le film intitulé 'Sharkenstein' ne mérite plus de figurer dans la base de données.

#### ✅ RÉSOLUTION


Il faut **supprimer** le film. Mais d'autres relations y font référence : on supprime d'abord ces lignes-là, puis le film.

**Étape 1 : trouver l'identifiant du film.**

```sql
SELECT tconst FROM Movies WHERE primaryTitle = 'Sharkenstein';
```

*Résultat dans la console H2 :*

```text
TCONST
tt5636234
(1 row)
```

**Étape 2 : supprimer les lignes qui font référence au film, puis le film lui-même.**

```sql
DELETE FROM Titles WHERE tconst = 'tt5636234';
```

*Résultat dans la console H2 :*

```text
(Update count: 7)
```

```sql
DELETE FROM Principals WHERE tconst = 'tt5636234';
```

*Résultat dans la console H2 :*

```text
(Update count: 4)
```

```sql
DELETE FROM Crews WHERE tconst = 'tt5636234';
```

*Résultat dans la console H2 :*

```text
(Update count: 1)
```

```sql
DELETE FROM Movies WHERE tconst = 'tt5636234';
```

*Résultat dans la console H2 :*

```text
(Update count: 1)
```

*Vérification :*

```sql
SELECT * FROM Movies WHERE primaryTitle = 'Sharkenstein';
```

*Résultat dans la console H2 :*

```text
TCONST | PRIMARYTITLE | RELEASEYEAR | RUNTIME | GENRES
(0 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- « Ne mérite plus de figurer dans la base » : on **supprime** des lignes, c'est un `DELETE` (partie 12 du cours).
- **Pourquoi ne pas supprimer directement le film ?** Parce que `Titles`, `Principals` et `Crews` contiennent des lignes qui font référence à ce film par leur clé étrangère `tconst`. Supprimer le film les laisserait pointer vers un film qui n'existe plus : H2 refuse (contrainte de référence, partie 10.3).

```sql
DELETE FROM Movies WHERE primaryTitle = 'Sharkenstein';
```

*Résultat dans la console H2 :*

```text
Erreur : Intégrité référentielle violation de contrainte: "CONSTRAINT_93: PUBLIC.TITLES FOREIGN KEY(TCONST) REFERENCES PUBLIC.MOVIES(TCONST) ('tt5636234')"
Referential integrity constraint violation: "CONSTRAINT_93: PUBLIC.TITLES FOREIGN KEY(TCONST) REFERENCES PUBLIC.MOVIES(TCONST) ('tt5636234')"; SQL statement:
DELETE FROM Movies WHERE primaryTitle = 'Sharkenstein' [23503-252]
```

- **L'ordre est donc l'inverse de celui du chargement** (question 1) : on supprime d'abord les lignes qui **font référence** (`Titles`, `Principals`, `Crews`), puis la ligne **référencée** (`Movies`).
- Les nombres affichés montrent ce qui a été supprimé : 7 traductions du titre, 4 acteurs, 1 équipe et 1 film.
- **Erreur grave à éviter :** oublier le `WHERE`. `DELETE FROM Titles;` viderait **toute** la relation.

---
# PARTIE C : requêtes de sélection

> Écrire les requêtes SQL permettant d'afficher les réponses aux questions suivantes.

**La méthode pour toutes les questions de cette partie** (partie 10.2 du cours) :
1. **Où sont les informations ?** Repérer les relations qui contiennent ce qu'on cherche et ce qu'on connaît.
2. **Quel chemin ?** Suivre les clés étrangères pour aller d'une relation à l'autre : chaque étape du chemin est un `JOIN ... ON ...`.
3. **Quelles lignes, quelles colonnes ?** Écrire le `WHERE`, puis choisir les colonnes du `SELECT`.

Les liens entre les relations :

| Relation | Attribut | Fait référence à |
|:---|:---|:---|
| Titles | `tconst` | `Movies.tconst` |
| Principals | `tconst` | `Movies.tconst` |
| Principals | `nconst` | `Names.nconst` |
| Crews | `tconst` | `Movies.tconst` |
| Crews | `director`, `writer` | `Names.nconst` |

---
# QUESTION 7

#### 📝 ÉNONCÉ


> 7. Dans combien de films 'Beatrix' est-il le prénom d'un des personnages ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Principals WHERE character LIKE '%Beatrix%';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
12
(1 row)
```

**Réponse : 12 films.**

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Où ?** Les personnages sont dans `Principals`, attribut `character`. Une seule relation suffit.
- **`LIKE` et pas `=` :** l'attribut `character` contient le nom **complet** du personnage, entre crochets et guillemets (`["Queen Beatrix"]`). On cherche donc le mot « Beatrix » **à l'intérieur** du texte :

```sql
SELECT character FROM Principals WHERE character LIKE '%Beatrix%';
```

*Résultat dans la console H2 :*

```text
CHARACTER
["Beatrix de Rijk first dutch female aviator"]
["Queen Beatrix"]
["Beatrix Carver"]
["Dr. Beatrix Cera"]
["Princeznou Beatrix"]
["Beatrix"]
["Beatrix"]
["Beatrix"]
["Beatrix Anderson"]
["The Bride","Beatrix Kiddo"]
["Beatrix"]
["Beatrix"]
(12 rows)
```

- **On compte des lignes, mais la question demande des films :** si un même film avait deux personnages appelés Beatrix, il serait compté deux fois. On vérifie que les 12 lignes correspondent bien à 12 films **différents**, avec `DISTINCT` (partie 7.5) :

```sql
SELECT DISTINCT tconst FROM Principals WHERE character LIKE '%Beatrix%';
```

*Résultat dans la console H2 :*

```text
TCONST
tt1927052
tt2241459
tt3479316
tt3543918
tt4122590
tt5212726
tt5299508
tt5814408
tt5897196
tt6019206
tt6385860
tt7708648
(12 rows)
```

---
# QUESTION 8

#### 📝 ÉNONCÉ


> 8. Combien de films Naomi Kawase a-t-elle réalisé depuis 2010 ?

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*)
FROM Crews
JOIN Names ON Crews.director = Names.nconst
WHERE Names.name = 'Naomi Kawase';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
8
(1 row)
```

**Réponse : 8 films.**

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Où ?** On connaît un **nom** (dans `Names`) et on cherche des films **réalisés** (dans `Crews`, attribut `director`).
- **Quel chemin ?** `Crews.director` fait référence à `Names.nconst` : une seule jointure, `ON Crews.director = Names.nconst`.
- **« Depuis 2010 » :** il n'y a pas de condition à ajouter, car la base ne contient **que** des films sortis depuis 2010 :

```sql
SELECT MIN(releaseYear) FROM Movies;
```

*Résultat dans la console H2 :*

```text
MIN(RELEASEYEAR)
2010
(1 row)
```

- Pour voir les films (on ajoute une troisième relation, `Movies`, pour avoir les titres) :

```sql
SELECT Movies.primaryTitle, Movies.releaseYear
FROM Crews
JOIN Names ON Crews.director = Names.nconst
JOIN Movies ON Crews.tconst = Movies.tconst
WHERE Names.name = 'Naomi Kawase'
ORDER BY Movies.releaseYear;
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE    | RELEASEYEAR
Genpin          | 2010
Hanezu          | 2011
Chiri           | 2012
Still the Water | 2014
Sweet Bean      | 2015
Radiance        | 2017
Vision          | 2018
True Mothers    | 2020
(8 rows)
```

---
# QUESTION 9

#### 📝 ÉNONCÉ


> 9. Quel est nom du réalisateur du film intitulé 'Okja' ?

#### ✅ RÉSOLUTION

```sql
SELECT Names.name
FROM Movies
JOIN Crews ON Movies.tconst = Crews.tconst
JOIN Names ON Crews.director = Names.nconst
WHERE Movies.primaryTitle = 'Okja';
```

*Résultat dans la console H2 :*

```text
NAME
Bong Joon Ho
(1 row)
```

**Réponse : Bong Joon Ho.**

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Où ?** On connaît un **titre** (dans `Movies`) et on cherche un **nom** (dans `Names`). Ces deux relations ne sont pas reliées directement : il faut passer par `Crews`.
- **Quel chemin ?** `Movies` → `Crews` (par `tconst`) → `Names` (par `director` = `nconst`). Deux étapes, donc **deux** `JOIN` (partie 10.2, étape 8 : trois tables).
- **Erreur fréquente :** écrire `ON Crews.tconst = Names.nconst`. Les deux attributs sont des identifiants, mais l'un désigne un **film** et l'autre une **personne** : la règle est fausse et le résultat est vide.

---
# QUESTION 10

#### 📝 ÉNONCÉ


> 10. Quels sont, dans l'ordre chronologique, les titres des films réalisés par Christopher Nolan depuis 2010 ?

#### ✅ RÉSOLUTION

```sql
SELECT Movies.primaryTitle, Movies.releaseYear
FROM Movies
JOIN Crews ON Movies.tconst = Crews.tconst
JOIN Names ON Crews.director = Names.nconst
WHERE Names.name = 'Christopher Nolan'
ORDER BY Movies.releaseYear;
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE          | RELEASEYEAR
Inception             | 2010
The Dark Knight Rises | 2012
Interstellar          | 2014
Dunkirk               | 2017
Tenet                 | 2020
Oppenheimer           | 2023
(6 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- Même chemin qu'à la question 9, mais dans l'autre sens : on connaît le **nom** et on cherche les **titres**.
- **« Dans l'ordre chronologique »** : on trie par année de sortie, de la plus ancienne à la plus récente, avec `ORDER BY Movies.releaseYear` (partie 7.4).
- Trier du **texte** qui contient des années fonctionne ici, parce que toutes les années ont **quatre chiffres** : `'2010'` vient bien avant `'2023'`. Avec des nombres de longueurs différentes, ce ne serait plus vrai (voir la question 13).
- Comme à la question 8, « depuis 2010 » ne demande aucune condition supplémentaire.

---
# QUESTION 11

#### 📝 ÉNONCÉ


> 11. Quels sont, dans l'ordre alphabétique, les principaux acteurs dont le titre français est 'Intouchables' ?

#### ✅ RÉSOLUTION

```sql
SELECT Names.name
FROM Titles
JOIN Principals ON Titles.tconst = Principals.tconst
JOIN Names ON Principals.nconst = Names.nconst
WHERE Titles.title = 'Intouchables' AND Titles.language = 'FR'
ORDER BY Names.name;
```

*Résultat dans la console H2 :*

```text
NAME
Anne Le Ny
Audrey Fleurot
François Cluzet
Omar Sy
(4 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Comprendre la question :** il s'agit des principaux acteurs **du film** dont le titre français est « Intouchables ».
- **Où ?** Le titre français est dans `Titles` (titre `'Intouchables'` et `language = 'FR'`), les acteurs principaux sont dans `Principals`, leurs noms dans `Names`.
- **Quel chemin ?** `Titles` → `Principals` (par `tconst`) → `Names` (par `nconst`). On n'a pas besoin de `Movies` : `Titles` et `Principals` ont toutes les deux l'identifiant du film.
- **Titre français :** comme à la question 3, l'attribut `language` contient ici le code du pays, `'FR'`. Le titre « Intouchables » existe pour ce film avec d'autres codes aussi (Canada, Inde…), d'où la condition `language = 'FR'` :

```sql
SELECT * FROM Titles WHERE title = 'Intouchables' AND language = 'FR';
```

*Résultat dans la console H2 :*

```text
TCONST    | ORDERING | TITLE        | LANGUAGE | ISORIGINAL
tt1675434 | 48       | Intouchables | FR       | 0
(1 row)
```

- « Dans l'ordre alphabétique » : `ORDER BY Names.name`. Le tri se fait sur le texte complet, donc d'après le **prénom**, puisque le nom complet commence par le prénom.

---
# QUESTION 12

#### 📝 ÉNONCÉ


> 12. Quelle est la liste alphabétique des titres, en français, des films dans lesquels Sandrine Kiberlain a joué ?

#### ✅ RÉSOLUTION

```sql
SELECT Titles.title
FROM Names
JOIN Principals ON Names.nconst = Principals.nconst
JOIN Titles ON Principals.tconst = Titles.tconst
WHERE Names.name = 'Sandrine Kiberlain' AND Titles.language = 'FR'
ORDER BY Titles.title;
```

*Résultat dans la console H2 :*

```text
TITLE
9 mois ferme
Aimer, boire et chanter
Amoureux de ma femme
Beur sur la ville
Comme un avion
Elle l'adore
Encore heureux
Fleuve noir
Floride
L'Envers du Décor
L'oiseau
La Belle et la Belle
...
(29 rows)
```

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Où ?** Le nom est dans `Names`, les rôles dans `Principals`, les titres français dans `Titles`.
- **Quel chemin ?** `Names` → `Principals` (par `nconst`) → `Titles` (par `tconst`). C'est le chemin de la question 11, parcouru dans l'autre sens.
- **29 titres, mais combien de films ?** Un même film peut avoir **plusieurs** titres français. Par exemple :

```sql
SELECT Titles.tconst, Titles.title FROM Titles WHERE Titles.title LIKE '%Alfred%' AND Titles.language = 'FR';
```

*Résultat dans la console H2 :*

```text
TCONST     | TITLE
tt12078024 | Les deux Alfred
tt12078024 | Les 2 Alfred
tt8333796  | Alfred Antoni: Un demi-siècle de cinéma en Alsace-Moselle
(3 rows)
```

  « Les deux Alfred » et « Les 2 Alfred » sont le **même** film (même `tconst`). Même chose pour « Mon Bébé » et « Mon bébé ». `DISTINCT` ne les fusionne pas, car les deux textes sont différents. En comptant les films eux-mêmes, on en trouve 24 :

```sql
SELECT DISTINCT Principals.tconst
FROM Names
JOIN Principals ON Names.nconst = Principals.nconst
JOIN Titles ON Principals.tconst = Titles.tconst
WHERE Names.name = 'Sandrine Kiberlain' AND Titles.language = 'FR';
```

*Résultat dans la console H2 :*

```text
TCONST
tt12078024
tt14371818
tt15115102
tt1533813
tt1805297
tt1830499
tt1964853
tt2246909
tt2354181
tt2391746
tt2403981
tt2751390
...
(24 rows)
```

- Les 24 films dans lesquels l'actrice a joué ont tous un titre français, puisque la base compte 24 rôles pour elle :

```sql
SELECT COUNT(*)
FROM Names
JOIN Principals ON Names.nconst = Principals.nconst
WHERE Names.name = 'Sandrine Kiberlain';
```

*Résultat dans la console H2 :*

```text
COUNT(*)
24
(1 row)
```

---
# PARTIE D : pour aller plus loin

> Écrire les requêtes SQL permettant d'afficher les réponses aux questions suivantes (celles-ci nécessiteront d'utiliser des fonctions non présentées dans le cours).

> ⚠️ **Programme :** les fonctions `CAST` et `LENGTH` (questions 13 et 14) et la sous-requête avec `IN` (question 15) ne sont **pas citées** par le programme officiel de Terminale, mais elles n'en sont **pas exclues** non plus. Le programme cite `SELECT`, `FROM`, `WHERE`, `JOIN`, `UPDATE`, `INSERT`, `DELETE`, et autorise `DISTINCT`, `ORDER BY` et les fonctions d'agrégation, « sans utiliser les clauses `GROUP BY` et `HAVING` ». C'est pour cela que le sujet présente cette partie comme un approfondissement.

---
# QUESTION 13

#### 📝 ÉNONCÉ


> 13. Quel est le titre et la durée du film le plus long sorti depuis 2010 ?

#### ✅ RÉSOLUTION

```sql
SELECT primaryTitle, runtime
FROM Movies
WHERE runtime <> '\N'
ORDER BY CAST(runtime AS INT) DESC;
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE                                                                                         | RUNTIME
Logistics                                                                                            | 51420
Ambiancé                                                                                             | 43200
Carnets Filmés (Liste Complète)                                                                      | 28643
Modern Times Forever                                                                                 | 14400
Qw                                                                                                   | 10062
The Birth: 10000 BC                                                                                  | 10000
Soldier                                                                                              | 7200
Hunger!                                                                                              | 6000
London EC1                                                                                           | 5460
Nieuwe Tieten                                                                                        | 4080
A 2nd generation film                                                                                | 3077
Saynatakuna: Masks                                                                                   | 2905
...
(149423 rows)
```

**Réponse : « Logistics », 51 420 minutes** (soit environ 35 jours et 17 heures).

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Le besoin :** trier les films par durée décroissante, puis lire la première ligne (méthode 1 de la partie 8 du cours, « obtenir une ligne entière »).
- **Premier piège : la durée est du texte.** Trier du texte, c'est le trier comme dans un dictionnaire, **caractère par caractère**. `'99'` passe alors **avant** `'51420'`, parce que le caractère `9` vient après le caractère `5`. Le tri par ordre décroissant met donc les films de 99 minutes en tête :

```sql
SELECT primaryTitle, runtime FROM Movies WHERE runtime <> '\N' ORDER BY runtime DESC;
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE                                                                                         | RUNTIME
#Realityhigh                                                                                         | 99
'71                                                                                                  | 99
05.RED.04.CLAY                                                                                       | 99
10.000 Km                                                                                            | 99
100 Days of Sunshine                                                                                 | 99
1098                                                                                                 | 99
11 A.M.                                                                                              | 99
12 Chòm Sao: Ve Duong Cho Yêu Chay                                                                   | 99
14 Days, 12 Nights                                                                                   | 99
1911                                                                                                 | 99
1st Strike                                                                                           | 99
2 Francs, 40 Pesetas                                                                                 | 99
...
(149423 rows)
```

- **La solution : `CAST(runtime AS INT)`** transforme le texte en **nombre entier** avant de trier. Les nombres sont alors comparés comme des nombres : 51 420 est bien plus grand que 99.
- **Deuxième piège : `'\N'`.** Les films dont la durée n'est pas renseignée contiennent `'\N'`, qui ne peut pas être transformé en nombre. Sans le `WHERE runtime <> '\N'`, H2 s'arrête sur une erreur :

```sql
SELECT primaryTitle, runtime FROM Movies ORDER BY CAST(runtime AS INT) DESC;
```

*Résultat dans la console H2 :*

```text
Erreur : Erreur lors de la conversion de données "\\N"
Data conversion error converting "\\N"; SQL statement:
SELECT primaryTitle, runtime FROM Movies ORDER BY CAST(runtime AS INT) DESC [22018-252]
```

- « Depuis 2010 » : comme aux questions 8 et 10, toute la base est concernée.

---
# QUESTION 14

#### 📝 ÉNONCÉ


> 14. Quel est la durée du film qui a le titre le plus long ?

#### ✅ RÉSOLUTION

```sql
SELECT runtime, LENGTH(primaryTitle)
FROM Movies
ORDER BY LENGTH(primaryTitle) DESC;
```

*Résultat dans la console H2 :*

```text
RUNTIME | CHAR_LENGTH(PRIMARYTITLE)
239     | 242
84      | 208
52      | 180
\N      | 179
212     | 152
82      | 149
70      | 142
\N      | 136
84      | 133
\N      | 133
97      | 129
118     | 128
...
(194207 rows)
```

**Réponse : 239 minutes.** Le titre le plus long compte 242 caractères : *I Saw a Little Bird Flying Over a Psychiatric Hospital Near Milan 2 Days Ago and Now it Became So Good that I Drank the Bordeaux Champagne of the Sample in 1771 So That My Soul Would Experience an Incredible Life in This Brutal and Gray World*.

#### 💡 EXPLICATION PÉDAGOGIQUE


- **La fonction `LENGTH(texte)`** donne le **nombre de caractères** d'un texte. On trie les films selon la longueur de leur titre, du plus long au plus court, et on lit la première ligne.
- H2 affiche `CHAR_LENGTH(PRIMARYTITLE)` en haut de la colonne : c'est le nom qu'il donne en interne à `LENGTH`, et c'est le même calcul.
- **Variante en une requête,** avec une sous-requête et `MAX` (méthode 2 de la partie 8 du cours) : on cherche d'abord la plus grande longueur de titre, puis le film qui a cette longueur. Le résultat est directement la réponse :

```sql
SELECT runtime
FROM Movies
WHERE LENGTH(primaryTitle) = (SELECT MAX(LENGTH(primaryTitle)) FROM Movies);
```

*Résultat dans la console H2 :*

```text
RUNTIME
239
(1 row)
```

- On n'affiche pas le titre dans le résultat parce qu'il est trop long : la console le couperait.

---
# QUESTION 15

#### 📝 ÉNONCÉ


> 15. Quels sont les castings des films dans lesquels a joué Scarlett Johansson ?

#### ✅ RÉSOLUTION

```sql
SELECT Movies.primaryTitle, Names.name
FROM Movies
JOIN Principals ON Movies.tconst = Principals.tconst
JOIN Names ON Principals.nconst = Names.nconst
WHERE Movies.tconst IN (SELECT Principals.tconst
                        FROM Principals
                        JOIN Names ON Principals.nconst = Names.nconst
                        WHERE Names.name = 'Scarlett Johansson')
ORDER BY Movies.primaryTitle, Names.name;
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE                        | NAME
Black Widow                         | David Harbour
Black Widow                         | Florence Pugh
Black Widow                         | Rachel Weisz
Black Widow                         | Scarlett Johansson
Captain America: Civil War          | Chris Evans
Captain America: Civil War          | Robert Downey Jr.
Captain America: Civil War          | Scarlett Johansson
Captain America: Civil War          | Sebastian Stan
Captain America: The Winter Soldier | Chris Evans
Captain America: The Winter Soldier | Robert Redford
Captain America: The Winter Soldier | Samuel L. Jackson
Captain America: The Winter Soldier | Scarlett Johansson
...
(73 rows)
```

**Réponse :** 73 lignes, une par acteur ou actrice principal de chacun des 17 films.

#### 💡 EXPLICATION PÉDAGOGIQUE


- **Comprendre la question :** le **casting** d'un film, c'est la liste de ses acteurs. On veut donc, pour **chaque** film où joue Scarlett Johansson, **tous** les acteurs de ce film, et pas seulement elle.
- **Le besoin, en deux temps :**
  1. trouver les films où elle a joué : c'est une jointure comme à la question 12 ;
  2. afficher tous les acteurs **de ces films-là**.
- **Étape 1 seule :** elle a joué dans 17 films de la base.

```sql
SELECT Movies.primaryTitle
FROM Movies
JOIN Principals ON Movies.tconst = Principals.tconst
JOIN Names ON Principals.nconst = Names.nconst
WHERE Names.name = 'Scarlett Johansson'
ORDER BY Movies.primaryTitle;
```

*Résultat dans la console H2 :*

```text
PRIMARYTITLE
Black Widow
Captain America: Civil War
Captain America: The Winter Soldier
Chef
Don Jon
Ghost in the Shell
Her
Hitchcock
Jojo Rabbit
Lucy
Marriage Story
Rough Night
...
(17 rows)
```

- **Étape 2 : la sous-requête avec `IN`.** La requête de l'étape 1 donne une **liste** de films (17 identifiants), et non une seule valeur. On ne peut donc pas écrire `tconst = (SELECT ...)` comme dans la partie 8 du cours : après un `=`, la sous-requête doit donner **une seule** valeur. `IN` signifie « fait partie de la liste » : `Movies.tconst IN (SELECT ...)` garde les films dont l'identifiant fait partie des 17.
- La requête principale est une jointure `Movies` → `Principals` → `Names`, comme à la question 11. Elle affiche chaque film avec ses acteurs, triés par titre puis par nom.
- Scarlett Johansson apparaît dans chaque casting : c'est normal, elle fait partie des acteurs de ces films.